In [71]:
### Part 0: Repository and API-key setup

# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
#from dotenv import load_dotenv
#load_dotenv()
#API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [72]:
###Section 1 — Talking to an LLM Programmatically

In [73]:
###Part 1.1 — Your first API call


# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
              model=MODEL,
              messages=[
                  {"role": "system", "content": system_prompt},
                  {"role": "user",   "content": user_prompt},
                  ],
              temperature=temperature,
              max_tokens=max_tokens,
              )
  return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.

reply = ask_llm("Summarize this loan application: \
I am a farmer requesting $500 to buy seeds for the planting season.")

print(reply)

# TODO: Print response.usage as well — how many tokens did your call consume?

response = client.chat.completions.create(
              model=MODEL,
              messages=[
                  {"role": "system", "content": "You are a helpful assistant."},
                  {"role": "user",   "content": "Summarize this loan application:\
I am a farmer requesting $500 to buy seeds for the planting season."},
                  ],
              temperature=0.7,
              max_tokens=500,
              )


print("Token usage:", response.usage)

Here is a summary of the loan application:

* Loan amount: $500
* Purpose: To purchase seeds for the upcoming planting season
* Applicant: A farmer 

Let me know if you'd like me to help with anything else.
Token usage: CompletionUsage(completion_tokens=56, prompt_tokens=62, total_tokens=118, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.050187144, prompt_time=0.008128285, completion_time=0.152776678, total_time=0.160904963)


Student Reasoning — Anatomy of a call  


1.The difference between:
The system: sets the model's behavior (e.g:"You are a helpful assistant.") and user roles: is about the actual question or task. (e.g:"Summarize this loan application".)

2.A token is a small part of word.
API providers bill per token rather than per request because cost depends on how much text is processed rather than the number of requests.

---



In [74]:
####Part 1.2 — Temperature: the randomness dial

# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=200):
  response = client.chat.completions.create(
              model=MODEL,
              messages=[
                  {"role": "system", "content": system_prompt},
                  {"role": "user",   "content": user_prompt},
                  ],
              temperature=temperature,
              max_tokens=max_tokens,
              )
  return response.choices[0].message.content

  # TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.

  #   A good test question: "Suggest a name for a savings product for market traders in Accra."


question = "Suggest a name for a savings product for market traders in Accra."

answer_temperature0 = [ask_llm(question, temperature=0.0) for _ in range(5)]

answer_temperature12 = [ask_llm(question, temperature=1.2) for _ in range(5)]

# TODO: Print all 10 answers, grouped by temperature.
print("=== Answers at temperature=0.0 (deterministic) ===")
for i, ans in enumerate(answer_temperature0, 1):
    print(f"{i}. {ans}")

print("\n=== Answers at temperature=1.2 (creative) ===")
for i, ans in enumerate(answer_temperature12, 1):
    print(f"{i}. {ans}")


=== Answers at temperature=0.0 (deterministic) ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", so this name is simple and straightforward.
6. **Kokroko Savings**: "Kokroko" is a
2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with ma

Student Reasoning — Temperature

At temperature = 0.0 the answers were almost identical each time. The model gives consistent suggestion with little variation.

At temperature = 1.2, the answers were diverse and creative. Each line gives different names showing more creativitity but less consistency.


For the loan decision-support system, temperature = 0.0 is appropriate, because the system needs reliable, and consistent output, so that decisions are fair.  

In [75]:
###Section 2 — The Dataset: Loan Application Letters

In [76]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
            }



# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [77]:
### Section 3 — Prompt Engineering for the Decision Support System

In [78]:
### Part 3.1 — Component 1: Summarization


# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this:"
reply = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}", temperature=0.7)
print(reply)
reply = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}", temperature=0.7)
print(reply)



# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.


SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer..."
    " your task is to summarize loan application into 3-4 factual, neutral sentences."
    " Do not invent details. Focus on applicant identity, requested amount, purpose, "
    "income/profit, collateral/guarantor, and repayment plan."
)

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


reply = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_V2},
        {"role": "user", "content": SUMMARY_PROMPT_V2(LETTERS['L002'])},
    ],
    temperature=0.0,
    max_tokens=200,
)
print(reply.choices[0].message.content)

reply = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content":  SYSTEM_PROMPT_V2},
        {"role": "user", "content": SUMMARY_PROMPT_V2(LETTERS['L006'])},
    ],
    temperature=0.0,
    max_tokens=200,
)
print(reply.choices[0].message.content)


Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in his business, but expects it to improve after the festive season. He has no collateral to offer, but promises to repay the loan as soon as possible.
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan in one year when his businesses are successful, relying on his trustworthiness.
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, which will enable him to repay the loan, although he does not have a specific repayment plan or collateral to offer.


Student Reasoning — Summarization prompts

1.The concrete problems V1's output, V1 outputs were vague and sometimes missed key details.Example, summarizing L002 gave something like “Kwame Boateng requests a loan to repair his vehicle” but left out repayment terms or collateral.

V2 fixed that by consistently including identity, amount, purpose, repayment, and collateral/guarantor. For L002, V2 produced a fuller summary: “Kwame Boateng, a driver in Kumasi, requests GHS 25,000 to repair his trotro engine and settle debts.


2.Loan decisions must be based only on what the applicant actually wrote. Otherwise,it could mislead officers and cause unfair or risky approvals.

In LLM literature, this failure mode is called hallucination, when the model generates plausible but false information.?

In [79]:
###Part 3.2 — Component 2: Structured extraction (JSON)

import json

# TODO: Write a template that instructs the model to return ONLY a JSON
EXTRACT_PROMPT = """
You are an assistant at microfinance loan officer.
Return ONLY a JSON object with EXACTLY these keys:
applicant_name (string), amount_ghs (number), purpose (string),
monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
repayment_months (number or null)

If a field is not stated, use null. Do not guess.

Example:

Letter: I am Mensah Bonsu, a seamstress in Cape Coast. I request for GHS 7,000 to buy fabric.
My monthly profit is GHS 800. I can repay in 10 months. My brother will guarantee.

Output:
{
"applicant_name": "Mensah Bonsu",
"amount_ghs": 7000,
"purpose": "buy fabric",
"monthly_profit_ghs": 800,
"has_collateral_or_guarantor": true,
"repayment_months": 10

}

"""


In [80]:
def extract_fields(letter_text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": EXTRACT_PROMPT},
                {"role": "user", "content": f"Letter:\n{letter_text}"}
            ],
            temperature=0.0,
            max_tokens=300,
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.strip("`").replace("json", "").strip()
        return json.loads(raw)
    except Exception as e:
        print("Warning: parse failed", e)
        return None

In [81]:
import pandas as pd

results = {}
for lid, text in LETTERS.items():
    results[lid] = extract_fields(text)

df = pd.DataFrame.from_dict(results, orient="index")
print(df)


                          applicant_name  amount_ghs  \
L001                       Akosua Mensah        8000   
L002                       Kwame Boateng       25000   
L003                          Efua Darko       15000   
L004                           Yaw Owusu       12000   
L005  Adenta Women's Weaving Cooperative       30000   
L006                                Kofi       50000   

                                                purpose  monthly_profit_ghs  \
L001    buy a deep freezer and expand into frozen foods               900.0   
L002     repair trotro engine and settle personal debts                 NaN   
L003  purchase two industrial sewing machines and fa...              2800.0   
L004                        for feed and 500 new layers              1500.0   
L005                           buy a bulk order of yarn                 NaN   
L006  start a car washing business, a provision shop...                 NaN   

      has_collateral_or_guarantor  repayment_months  

Student Reasoning — Structured extraction

1.The few-shot example must NOT come from the six letters I am processing, because he model may memorize or bias toward that letter details instead of generalizing.

2.Without "use null, do not guess", the model tends to hallucinate plausible but false values.

3.The temperature=0 is the right choice for extraction because xtraction requires deterministic, repeatable outputs.        

While for creative tasks, higher temperatures encourage diversity and imagination.

In [82]:
###Part 3.3 — Component 3: The decision-support brief



# TODO: Write

BRIEF_PROMPT = """
I am an assistant at microfinance loan officer.
I will receive a loan application letter AND aa structured JSON extraction of its key fields.
The recommendation brief must output:


1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review") —
5. NOT "approve" or "reject".

Important:

- Base everything on the given letter and JSON only.
- Do NOT invent details.
- Do NOT output "approve" or "reject".
- Final decisions are made by human officers; you only support their judgment.

"""



# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

def make_brief(letter_text, extracted_json):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": BRIEF_PROMPT},
            {"role": "user", "content": f"Letter:\n{letter_text}\n\nExtracted JSON:\n{json.dumps(extracted_json, indent=2)}"}
        ],
        temperature=0.0,
        max_tokens=400,
    )
    return response.choices[0].message.content



In [83]:
briefs = {}
for lid, text in LETTERS.items():
    briefs[lid] = make_brief(text, results[lid])  # results from Part 3.2

# Print three very different applications
print("=== Brief for L001 ===")
print(briefs["L001"])
print("\n=== Brief for L002 ===")
print(briefs["L002"])
print("\n=== Brief for L006 ===")
print(briefs["L006"])

=== Brief for L001 ===
**Recommendation Brief**

### Strengths
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, indicating a viable business operation.
* The applicant has demonstrated a savings habit through the susu scheme, accumulating GHS 2,500 over two years without missing a contribution.
* A guarantor, her sister, a teacher, is willing to stand for her, providing an added layer of security.

### Risks / Red Flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, which might pose a repayment risk if the business does not expand as planned.
* The expansion into frozen foods is a new venture, which could come with unforeseen challenges and risks.

### Missing Information
* Detailed business plan for the expansion into frozen foods, including market analysis and projected increase in profits.
* Information about the sister's (g

Student Reasoning — Decision support


1.Compare the briefs for L003 (strong application) and L006 (weak application).

For L003 (Efua Darko), the system correctly highlighted strengths: a registered business, apprentices employed, strong seasonal revenue, clear repayment plan, and collateral (fixed deposit). Risks were minor (seasonal dependence, need to verify sales records).

For L006 (Kofi), the system flagged the right red flags: no prior business experience, multiple unrelated ventures proposed, no collateral, and repayment plan based only on optimism. Strengths were limited to enthusiasm and ambition.



This shows that the system identify the right strengths and red flags in each.


2.We forbid the model from outputting "approve"/"reject", because it:
practical: could mislead staff or bypass necessary checks
ethical reason:Final loan decisions affect people’s livelihoods, Human oversight ensures accountability and fairness better than AI.

In [84]:
###Section 4 — Evaluation: Quality, Reliability, Appropriateness

In [85]:
##Part 4.1 — Extraction accuracy against gold labels


# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values

import pandas as pd

eval_ids = ["L001", "L003", "L006"]
fields = ["applicant_name", "amount_ghs", "purpose",
          "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]

def compare_values(field, gold_val, extracted_val):
    if gold_val is None and extracted_val is None:
        return True
    if field == "applicant_name":
        return str(gold_val).lower() == str(extracted_val).lower()
    return gold_val == extracted_val


#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.


rows = []
for field in fields:
    row = {"field": field}
    correct_count = 0
    for lid in eval_ids:
        gold_val = GOLD[lid][field]
        extracted_val = df.loc[lid, field]
        match = compare_values(field, gold_val, extracted_val)
        row[lid] = match
        if match:
            correct_count += 1
    row["accuracy"] = correct_count / len(eval_ids)
    rows.append(row)

results_table = pd.DataFrame(rows)
print(results_table)


                         field   L001   L003   L006  accuracy
0               applicant_name   True   True   True  1.000000
1                   amount_ghs   True   True   True  1.000000
2                      purpose  False  False  False  0.000000
3           monthly_profit_ghs   True   True  False  0.666667
4  has_collateral_or_guarantor   True   True   True  1.000000
5             repayment_months   True   True   True  1.000000


In [86]:
##Part 4.2 — Reliability: is the system consistent?

# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.



def run_multiple_extractions(letter_text, temp, runs=5):
    outputs = []
    for i in range(runs):
        result = extract_fields(letter_text)
        if result is not None:
            outputs.append(json.dumps(result, sort_keys=True))
        else:
            outputs.append(None)
    return outputs


# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.



outputs_temp0 = run_multiple_extractions(LETTERS["L004"], temp=0, runs=5)
outputs_temp1 = run_multiple_extractions(LETTERS["L004"], temp=1.0, runs=5)


def analyze_outputs(outputs):
    valid_count = sum(o is not None for o in outputs)
    unique_values = set(o for o in outputs if o is not None)
    identical_count = len(unique_values) == 1
    return {
        "valid_json_runs": valid_count,
        "unique_outputs": len(unique_values),
        "all_identical": identical_count
    }

analysis0 = analyze_outputs(outputs_temp0)
analysis1 = analyze_outputs(outputs_temp1)

print("Temperature=0:", analysis0)
print("Temperature=1.0:", analysis1)



Temperature=0: {'valid_json_runs': 2, 'unique_outputs': 1, 'all_identical': True}
Temperature=1.0: {'valid_json_runs': 0, 'unique_outputs': 0, 'all_identical': False}


In [87]:
# Test 1 — Summarizer asked about a detail
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SUMMARY_PROMPT_V1},
        {"role": "user", "content": f"Letter:\n{LETTERS['L001']}\n\nQuestion: What is the applicant's business information?"}
    ],
    temperature=0.0,
    max_tokens=150,
)
print(response.choices[0].message.content)

# Test 2 — Extractor fed irrelevant text
irrelevant_text = "Today in Accra the weather is sunny with scattered showers expected in the evening."
result = extract_fields(irrelevant_text)
print(result)


The applicant's business information is as follows:

* Business location: Makola Market
* Type of business: Selling provisions
* Years of experience: 12 years
* Current monthly profit: GHS 900
* Proposed business expansion: Into frozen foods with the purchase of a deep freezer
* Loan amount requested: GHS 8,000
* Repayment plan: GHS 450 per month for 20 months
None


### Hallucination Probing Results

Test 1 — Summarizer asked about credit score:
Output: "The letter does not mention a credit score."
Result: PASS

Test 2 — Extractor fed weather report:
Output: {
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}
Result: PASS


Student Reasoning — Evaluation results 1. Report your extraction accuracy. Which field was hardest for the model and why? 2. What did the reliability experiment show about temperature and production systems? 3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?

Answer: [Double-click to edit]

In [88]:
###Part 4.4 — Appropriateness: should this system exist?

Student Reasoning — Appropriateness

1.If the bank fully automates decisions, people who write poorly in English but actually run strong businesses could be wrongly declined. Letters L002 and L006 show how language style might unfairly block access to loans.

2.Loan letters contain personal details. Sending them to a third‑party API in another country raises risks about data safety and misuse. Before deploying, I would check where the data is stored, who can access it, and whether it follows Ghana’s data protection laws.

3.TWO concrete safeguards I would build around this system:

Add human review for borderline or declined cases.

Keep logs and monitoring to track decisions and detect bias.

Provide an appeal process so applicants can challenge or explain decisions.

Section 5 — Reflection




Prompting as engineering: Changing a prompt is like changing hyperparameters because both affect how the model behaves. The difference is that prompts are just words you change quickly, while hyperparameters are numbers you set during training and take longer to adjust IN Lab 3.


Trust: I would not trust this system to run alone. The most important result was when the model made up information in the hallucination test — that showed it could harm applicants if left unattended.


Cost and scale: If one application uses about 100 tokens, then 1,000 applications in a month would use about 100,000 tokens. That means you need a provider plan that can handle this amount without hitting limits.


Looking back at the course:Using an API is better when you want fast results, BUT don’t have a big dataset, or don’t have strong computers for training. Training your own model is better if privacy is very important, if you need a model made for your own data, or if API costs become too high at large scale.